In [ ]:
import pandas as pd
from bert_score import score
import torch
import random
import numpy as np
import os

def set_seed(seed_value=42):
    """
    Fungsi untuk mengunci semua random seed agar eksperimen reproducible.
    """
    # 1. Kunci seed untuk library dasar Python dan environment
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    # 2. Kunci seed untuk NumPy
    np.random.seed(seed_value)
    
    # 3. Kunci seed untuk PyTorch (CPU & GPU)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        
        # 4. Paksa operasi CuDNN agar deterministik
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def calculate_bertscore(input_csv, output_csv):
    print(f"Membaca file {input_csv}...")
    # 1. Baca file CSV
    df = pd.read_csv(input_csv)

    # 2. Hapus baris kosong (jika ada) di kolom yang akan dievaluasi
    df = df.dropna(subset=['Essay', 'description'])

    # 3. Siapkan list teks kandidat (Essay) dan referensi (description)
    candidates = df['Essay'].tolist()
    references = df['description'].tolist()

    # Determine device: Use CUDA (GPU) if available, otherwise CPU
    def get_device():
        if torch.cuda.is_available(): return "cuda"
        try:
            if torch.xpu.is_available(): return "xpu"
        except: pass
        if torch.backends.mps.is_available(): return "mps"
        return "cpu"

    device = get_device()
    print(f"Menggunakan device: {device}")
    print(f"Menghitung BERTScore untuk {len(candidates)} baris. Ini mungkin memakan waktu beberapa saat...")

    # 4. Hitung BERTScore
    P, R, F1 = score(
        candidates, 
        references, 
        lang="en", 
        model_type="roberta-large", 
        verbose=True, 
        device=device
    )

    # 5. Tambahkan hasil ke dalam DataFrame
    # Mengubah tensor ke bentuk list/numpy array agar bisa disimpan di pandas
    df['bertscore_precision'] = P.cpu().numpy()
    df['bertscore_recall'] = R.cpu().numpy()
    df['bertscore_f1'] = F1.cpu().numpy()

    # 6. Simpan hasil perhitungan ke file CSV baru
    df.to_csv(output_csv, index=False)
    print(f"\nSelesai! Hasil perhitungan telah disimpan di: {output_csv}")

if __name__ == "__main__":
    # KUNCI SEED SEBELUM MENJALANKAN APAPUN
    set_seed(42) 
    
    input_file = "data_short_gemma.csv"
    output_file = "../short_gemma_bertscore.csv"

    calculate_bertscore(input_file, output_file)

Membaca file multimodal/data_short_gemma.csv...
Menggunakan device: xpu
Menghitung BERTScore untuk 1054 baris. Ini mungkin memakan waktu beberapa saat...


Loading weights: 100%|██████████| 389/389 [00:01<00:00, 335.71it/s, Materializing param=encoder.layer.23.output.dense.weight]              
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


100%|██████████| 19/19 [01:24<00:00,  4.43s/it]


computing greedy matching.


100%|██████████| 17/17 [00:16<00:00,  1.02it/s]


done in 100.98 seconds, 10.44 sentences/sec

Selesai! Hasil perhitungan telah disimpan di: multimodal/short_gemma_bertscore.csv
